
# 🐼 EDA con Pandas — comparación con NumPy

En esta práctica repetimos el **mismo análisis exploratorio (EDA)** que hicimos con NumPy,  
pero ahora usando **Pandas**, para ver cómo cambia la sintaxis, el tiempo de desarrollo  
y las capacidades de análisis.

---

## 🧾 Preparación del dataset

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

df = pd.DataFrame({
    "ID": np.arange(1, n+1),
    "Edad": rng.integers(18, 65, size=n),
    "Salario": rng.normal(2500, 800, size=n),
    "Experiencia": np.clip(rng.integers(0, 40, size=n), 0, None),
    "Horas": rng.normal(40, 5, size=n),
    "Nivel": rng.integers(0, 5, size=n),
    "Satisfacción": rng.normal(7, 2, size=n)
})

# Añadimos outliers y NaN
df.loc[rng.choice(n, 10, replace=False), "Salario"] *= 4
df.loc[rng.choice(n, 15, replace=False), "Satisfacción"] = np.nan

df_original = df.copy(deep=True)  # conservar los datos antes de limpiarlos
df.head(10)

---

## 🧠 Tarea 1: Descripción básica

In [ ]:
print(df.shape)
print(df.columns)
df.head()

✅ **Más directo**: Pandas ya muestra etiquetas y formato de tabla.

---

## 📏 Tarea 2: Tipos y rangos

In [ ]:
print(df.dtypes)
df[["Edad", "Salario"]].agg(["min", "max", "mean", "median", "std"])

✅ En una sola línea obtenemos estadísticas básicas.

---

## 💭 Tarea 3: Filtrado de datos

In [ ]:
# 1. Salarios > 4000 €
print(df[df["Salario"] > 4000])

# 2. Más de 45h y >10 años experiencia
print(df[(df["Horas"] > 45) & (df["Experiencia"] > 10)][["ID", "Edad", "Horas", "Experiencia"]])

# 3. Porcentaje nivel ≥ 3
print((df["Nivel"] >= 3).mean() * 100)

✅ Sintaxis más legible y declarativa.

---

## 🧹 Tarea 4: Limpieza — valores faltantes

In [ ]:
print(df["Satisfacción"].isna().sum())
df["Satisfacción"] = df["Satisfacción"].fillna(df["Satisfacción"].mean())
print(df["Satisfacción"].isna().any())


✅ Un único método `fillna()` evita usar máscaras.

---

## ⚠️ Tarea 5: Detección de outliers

In [ ]:
Q1 = df["Salario"].quantile(0.25)
Q3 = df["Salario"].quantile(0.75)
IQR = Q3 - Q1

filtro = (df["Salario"] < Q1 - 1.5*IQR) | (df["Salario"] > Q3 + 1.5*IQR)
df[filtro]

✅ `quantile()` y condiciones combinadas simplifican la detección.

---

## 🧰 Tarea 6: Sustitución de outliers

In [ ]:
# Ejemplo de winsorización: limitar ambos extremos sin elevar los salarios bajos.
p05, p95 = df["Salario"].quantile([0.05, 0.95])
df["Salario"] = df["Salario"].clip(lower=p05, upper=p95)


La winsorización limita los extremos a los percentiles 5 y 95. Es una decisión didáctica, no una regla universal: antes de tratar un valor extremo hay que comprobar si es un error o un caso real. Se conserva `df_original` para comparar.

---

## 📊 Tarea 7: Agrupaciones y resúmenes simples


In [ ]:
# Media por nivel
print(df.groupby("Nivel")["Salario"].mean())
# Satisfacción media por grupo de edad
bins = [0, 30, 50, 100]
labels = ["≤30", "31–50", ">50"]
df["GrupoEdad"] = pd.cut(df["Edad"], bins=bins, labels=labels)
print(df.groupby("GrupoEdad", observed=True)["Satisfacción"].mean())

✅ `groupby()` y `cut()` hacen triviales las operaciones por grupo.

---

## 🧩 Tarea 8: Normalización y escalado

In [ ]:
df["Salario_norm"] = (df["Salario"] - df["Salario"].min()) / (df["Salario"].max() - df["Salario"].min())
df["Horas_z"] = (df["Horas"] - df["Horas"].mean()) / df["Horas"].std()
df[["Salario_norm", "Horas_z"]].describe()

✅ Con Pandas, las operaciones vectorizadas siguen siendo de NumPy,
pero la sintaxis es más limpia y las columnas se guardan fácilmente.

---

## 🧮 Tarea 9: Correlaciones simples

In [ ]:
print(df[["Salario", "Experiencia"]].corr())
print(df[["Edad", "Satisfacción"]].corr())

✅ `corr()` calcula la matriz de correlación directamente.

---

## 🧾 Tarea 10: Estadísticas generales

In [ ]:
df.describe(include="all")

✅ Una sola línea genera resumen completo de medias, desvíos,
mínimos, máximos y cuantiles — con una interfaz tabular; NumPy también dispone de funciones de resumen.

---

## ⚖️ Comparación final: NumPy vs Pandas

| Aspecto                       | NumPy                                     | Pandas                                         |
| ----------------------------- | ----------------------------------------- | ---------------------------------------------- |
| **Estructura**                | Arrays de un mismo dtype; también texto y tipos estructurados          | DataFrames con columnas heterogéneas           |
| **Acceso a datos**            | Por índice numérico                       | Por nombre de columna                          |
| **Limpieza (NaN, outliers)**  | Manual, usando máscaras y `np.isnan()`    | Métodos directos (`fillna`, `replace`, `clip`) |
| **Agrupaciones**              | Manual, por filtrado repetido             | `groupby`, `cut`, `pivot_table`                |
| **Resúmenes**                 | Funciones separadas (`mean`, `std`, etc.) | `describe()`, `agg()` integrados               |
| **Eficiencia**                | Rendimiento según operación, dtype y datos                  | Rendimiento según operación, dtype y datos               |
| **Lectura de datos externos** | `loadtxt`, `genfromtxt`, `load` (entre otros) | CSV, Excel, SQL, Parquet, JSON…                |

---

## 🧠 Reflexión: cuándo usar cada uno

* 🧮 **NumPy**: ideal para **cálculos numéricos puros**, operaciones de álgebra, vectores, tensores o machine learning a bajo nivel.
* 🐼 **Pandas**: ideal para **análisis de datos reales** con valores faltantes, mezcla de tipos, y manipulación de columnas o tablas grandes.

En la práctica, **Pandas usa NumPy por debajo** para los cálculos numéricos,
pero aporta una capa de **semántica tabular y limpieza** que simplifica enormemente el trabajo.

---

## Tipos de datos anulables y PyArrow (ampliación opcional)

Pandas puede almacenar columnas con arrays NumPy, tipos anulables propios o tipos Arrow. Esto no equivale a cambiar un motor global de ejecución. No existe la opción `mode.dtype_backend`.

Para convertir una copia se usa `convert_dtypes(dtype_backend=...)` (pandas ≥ 2.0). La variante `pyarrow` requiere instalar PyArrow. No hace falta para el EDA de esta clase.

Referencia: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html


In [ ]:
# Tipos anulables de pandas: no requieren PyArrow.
df_nullable = df.convert_dtypes()
print(df_nullable.dtypes)


Conversión opcional a tipos Arrow, manteniendo el DataFrame de trabajo intacto:


In [ ]:
try:
    import pyarrow
except ImportError:
    print("PyArrow no está instalado; continuamos con pandas sin esta ampliación.")
else:
    df_arrow = df.convert_dtypes(dtype_backend="pyarrow")
    print(df_arrow.dtypes)


Arrow aporta tipos anulables e interoperabilidad con otras herramientas. Su consumo de memoria y rendimiento dependen de los datos y las operaciones: no siempre mejora a NumPy.

## Próximo paso

Compara la sintaxis y los resultados con el EDA de NumPy. Una semilla idéntica con generadores distintos no produce los mismos valores; para una comparación exacta hay que compartir los datos, el generador y el orden de las llamadas. NumPy usa por defecto `ddof=0` para la desviación típica; pandas usa `ddof=1`.

Después de introducir Matplotlib, continúa con `03_pandas_eda_visualizacion.ipynb`. Este cuaderno se centra en el análisis sin gráficas.
